# 03: Local LLM
    แบบฝึกหัดสำหรับผู้เข้าร่วม Sri Trang

    เลือก Python kernel แล้วกด **Run All** cell แรกจะติดตั้ง packages ให้
    สำหรับการรัน local ให้เปิด Ollama ที่มีโมเดลตาม README ไว้ก่อน
    โค้ด HTTP API ใช้ได้ทั้ง local และ Colab แต่ `localhost` ใน Colab หมายถึงเครื่อง Colab
    หากเรียก Ollama จาก Colab ต้องกำหนด HOST เป็น endpoint ที่ runtime เข้าถึงได้
    โค้ดและข้อมูลทั้งหมดอยู่ใน cells ไม่ต้องเตรียมหรือแก้ไฟล์ประกอบ
    ตัวอย่างทั้งหมดเป็นข้อมูลสมมติสำหรับการเรียน ไม่ใช่ข้อมูลหรือ SOP จริงของบริษัท


## ติดตั้ง packages
รันใน Python kernel ได้ทั้ง Jupyter และ Colab


In [ ]:
%pip install -q "requests>=2.32,<3"


## 1. เชื่อมต่อ local model
กำหนดโมเดลและฟังก์ชันส่งข้อความไปยัง Ollama บนเครื่องนี้


In [ ]:
import requests

HOST = "http://localhost:11434"
MODEL = "qwen3:4b"

def chat(messages, **kwargs):
    payload = {"model": MODEL, "messages": messages, "stream": False,
               "think": False, "options": {"temperature": 0}, **kwargs}
    response = requests.post(HOST + "/api/chat", json=payload, timeout=180)
    return response.json()["message"]


## 2. เตรียมรายงานกะ
System prompt กำหนดหน้าที่ ส่วน user prompt ให้ข้อมูลที่ต้องสรุป


In [ ]:
system_prompt = "คุณเป็นผู้ช่วยสรุปรายงานโรงงานยาง สรุปภาษาไทย 3 ข้อ ใช้เฉพาะข้อมูลที่ได้รับ"
user_prompt = """รายงานกะเช้า (สมมติ): รับน้ำยาง 12 ตัน รถขนส่งล่าช้า 30 นาที
ทีม QC กำลังตรวจตัวอย่างล็อต L-101 ยังไม่มีผลตรวจ จึงยังไม่ปล่อยล็อตนี้"""


## 3. ส่ง prompt และอ่านคำตอบ
โมเดล local สร้างสรุปจากรายงานที่ให้


In [ ]:
messages = [{"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}]
answer = chat(messages)
print(answer["content"])


## 4. เปลี่ยนรูปแบบคำตอบ
ใช้รายงานเดิม แต่ให้สรุปเพียงหนึ่งประโยค


In [ ]:
short_answer = chat([
    {"role": "system", "content": "สรุปรายงานภาษาไทยหนึ่งประโยค ใช้เฉพาะข้อมูลที่ได้รับ"},
    {"role": "user", "content": user_prompt},
])
print(short_answer["content"])


## 5. สรุปสิ่งที่ได้เรียน
Prompt เดียวกันสามารถส่งไปยัง cloud หรือ local model ได้ ส่วน system prompt ช่วยกำหนดรูปแบบคำตอบ หากมีคำตอบ cloud จากบทก่อน สามารถนำมาเปรียบเทียบภายหลังได้ ไม่จำเป็นต่อการรัน notebook นี้

ลองเปลี่ยนรายงานในขั้นตอน 2 แล้วกด Run All อีกครั้ง
